# Model Analysis

In this notebook we will analyse the robustness of this model against typical attacks. We will explore whether or not the use of uncertainty can make the model aware of slight changes in the input.

## Import libraries

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

: 

In [ ]:
from src.data import load_and_prepare_datasets
from src.utils import load_model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
model, tokenizer = load_model("models/distilbert_toxic", device=device)
model.eval()

## Data exploration

In [ ]:
datasets, id2label = load_and_prepare_datasets(
    tokenizer=tokenizer,
    max_length=128,
    val_size=0.1,
    test_size=0.1,
    seed=42
)
 
test_set = datasets["test"]
len(test_set)

In [ ]:
i = 4
sample = test_set[i]
 
text = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
label = sample["labels"]
 
print("TEXT:", text)
print("TRUE LABELS:", label.numpy())
print("CLASS NAMES:", id2label)

How to Read These F1-macro and F1-micro?
 
The metrics we got are completely **normal and healthy**, showing the model learned effectively without overfitting too much.
 
* The F1-micro score (around **0.79**) is strong and shows good **generalization**, it's performing well on new data it's never seen. The small difference between the training F1-micro (~0.87) and the validation F1-micro (~0.79) is good; it means the model is **stable** and not just memorizing the training examples.
 
* The F1-macro score (around **0.65**) is lower, but that is **expected** for the Jigsaw dataset. The F1-macro score is pulled down by the **imbalance** in the data, where super rare labels (like *threat* or *identity\_hate*) get low individual scores.
 
Here's how these two scores are calculated:
 
* **F1-micro:** It calculates the F1 score globally, counting up the total correct and incorrect predictions across all labels. It basically treats all predictions as a single, combined pool.
 
    $$
    \text{F1}_{\text{micro}} =
    \frac{2 \cdot \text{TP}}
        {\,2 \cdot \text{TP} + \text{FP} + \text{FN}\,}
    $$
 
 
* **F1-macro:** It calculates the F1 score for *each* of the six toxicity labels separately, and then takes a simple average of those six scores.
 
    $$
    \text{F1}_{\text{macro}} =
    \frac{1}{6}
    \sum_{i=1}^{6} \text{F1}_i
    $$